<a href="https://colab.research.google.com/github/kapilpandey09/NLP/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("kazanova/sentiment140")

100%|██████████| 80.9M/80.9M [00:01<00:00, 61.9MB/s]

Extracting files...


In [2]:
path

'/root/.cache/kagglehub/datasets/kazanova/sentiment140/versions/2'

In [3]:
# df = pd.read_csv(path + "/training.1600000.processed.noemoticon.csv", encoding="utf-8", encoding_errors="ignore")

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [5]:
import pandas as pd

df = pd.read_csv(path +
    "/training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None,
    names=[
        "sentiment",
        "id",
        "date",
        "query",
        "user",
        "text"
    ]
)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df["sentiment"].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
df = df[["sentiment", "text"]]

In [ ]:
df["sentiment"].value_counts()

In [ ]:
df["sentiment"].nunique()

In [ ]:
df

In [ ]:
df["sentiment"] = df["sentiment"].replace({4:1})

In [ ]:
df

In [ ]:
df["sentiment"].value_counts().plot(kind="bar")

In [ ]:
df.duplicated().sum()

In [ ]:
df["text"].duplicated().sum()

In [ ]:
# df["char_count"] = df["text"].str.len()
# df["char_count"].describe()

In [ ]:
# df["word_count"] = df["text"].str.split().str.len()
# df["word_count"].describe()

In [ ]:
df[df["sentiment"]==1]["text"].sample(5)

In [ ]:
df[df["sentiment"]==0]["text"].sample(5)

In [ ]:
conflict = (
    df.groupby("text")["sentiment"]
      .nunique()
      .reset_index()
)

conflict[conflict["sentiment"] > 1]

In [ ]:
conflict

In [ ]:
# df = df[~df["text"].isin(conflict_texts)]

In [ ]:
df = (
    df.groupby("text")["sentiment"]
      .agg(lambda x: x.mode()[0])
      .reset_index()
)

In [ ]:
conflict = (
    df.groupby("text")["sentiment"]
      .nunique()
)

print("Conflicting texts:", (conflict > 1).sum())

In [ ]:
"Duplicate rows:", df.duplicated().sum()

In [ ]:
print((df["text"].str.strip() == "").sum())

In [ ]:
print(df["text"].nunique())

In [ ]:
df.info()

In [ ]:
df["sentiment"].value_counts().plot(kind="bar")


In [ ]:
df.isnull().sum()

In [ ]:
# negative_words = " ".join(
#     df[df["sentiment"]==0]["text"]
# )

In [ ]:
# positive_words = " ".join(
#     df[df["sentiment"]==1]["text"]
# )

In [ ]:
df["text"].str.contains("http").sum()

In [ ]:
df["text"].str.contains("@").sum()

In [ ]:
df["text"].str.contains("#").sum()

In [ ]:
df["text"].sample(20)

In [ ]:
import matplotlib.pyplot as plt

df["sentiment"].value_counts().sort_index().plot(
    kind="bar"
)

plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
(df["text"].str.strip() == "").sum()

In [ ]:
print(df.shape)
print(df["sentiment"].value_counts())
print(df["text"].nunique())

In [ ]:
df["length"] = df["text"].str.split().str.len()

df["length"].describe(percentiles=[0.5,0.9,0.95,0.99])

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["text"])

print(len(tokenizer.word_index))

In [ ]:
df["clean_text"] = df["text"]

In [ ]:
import re

def clean_text(text):

    # lowercase
    text = text.lower()

    # URLs -> placeholder
    text = re.sub(r'http\S+|www\S+', ' url ', text)

    # User mentions -> placeholder
    text = re.sub(r'@\w+', ' user ', text)

    # hashtags symbol remove, word preserve
    text = re.sub(r'#', '', text)

    # Positive emoticons
    text = re.sub(r'(:\)|:-\)|:d|=d|=\))', ' emo_pos ', text)

    # Negative emoticons
    text = re.sub(r'(:\(|:-\(|=\()', ' emo_neg ', text)

    # Common contractions
    text = re.sub(r"don't", "do not", text)
    text = re.sub(r"can't", "can not", text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)

    # Repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Keep only letters and underscore
    text = re.sub(r'[^a-zA-Z_\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
df[["text", "clean_text"]].head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_WORDS = 50000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
# X_train_seq

In [ ]:
# tokenizer.word_index

In [ ]:
tweet_lengths = df["clean_text"].str.split().apply(len)

print(tweet_lengths.describe())

In [ ]:
print(tweet_lengths.quantile([0.50, 0.75, 0.90, 0.95, 0.99]))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.hist(
    tweet_lengths,
    bins=50
)

plt.xlabel("Tweet Length")
plt.ylabel("Count")
plt.title("Tweet Length Distribution")

plt.show()

In [ ]:
MAX_LEN = 30

truncated = (tweet_lengths > MAX_LEN).sum()

percentage = (
    truncated / len(tweet_lengths)
) * 100

print(f"Truncated Tweets: {truncated}")
print(f"Percentage: {percentage:.2f}%")

In [ ]:

MAX_LEN = 30

from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"

)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"

)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.regularizers import l2


model = Sequential([
    Input(shape=(30,)),

    Embedding(
        input_dim=50000,
        output_dim=128,
        mask_zero=True
    ),

    Bidirectional(
        LSTM(128, dropout=0.2)
    ),

    Dropout(0.3),

    Dense(
    64,
    activation="relu",

    kernel_regularizer=l2(1e-4)
),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
print(X_train_pad.shape)
print(X_test_pad.shape)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=512,
    callbacks=[early_stop]
)

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Loss :", test_loss)
print("Test Accuracy :", test_accuracy)

In [ ]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

In [ ]:
len(y_pred)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
!wget https://nlp.stanford.edu/data/glove.twitter.27B.zip
!unzip glove.twitter.27B.zip

In [ ]:
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
embeddings_index = {}

with open(
    "glove.twitter.27B.100d.txt",
    encoding="utf8"
) as f:

    for line in f:
        values = line.split()

        word = values[0]

        coefs = np.asarray(
            values[1:],
            dtype="float32"
        )

        embeddings_index[word] = coefs

print("Loaded vectors:", len(embeddings_index))

In [ ]:
vocab_size = 50000
embedding_dim = 100

embedding_matrix = np.zeros(
    (vocab_size, embedding_dim)
)

for word, idx in tokenizer.word_index.items():

    if idx >= vocab_size:
        continue

    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[idx] = vector

print("Embedding matrix shape:")
print(embedding_matrix.shape)

In [ ]:
model = Sequential([

    Input(shape=(30,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=False,
        mask_zero=True
    ),

    Bidirectional(
        LSTM(
            128,
            dropout=0.2
        )
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(1e-4)
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=512,
    callbacks=[early_stop]
)

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Loss :", test_loss)
print("Test Accuracy :", test_accuracy)

In [ ]:
y_pred_prob = model.predict(X_test_pad)

y_pred = (
    y_pred_prob > 0.5
).astype(int)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

# Base line models

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

X = tfidf.fit_transform(df["clean_text"])
y = df["sentiment"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

In [ ]:
tweet = "I love this phone. Camera is amazing."

tweet = clean_text(tweet)

tweet_vector = tfidf.transform([tweet])

prediction = model.predict(tweet_vector)

print(prediction)

In [ ]:

# tfidf.vocabulary_

In [ ]:
len(tfidf.vocabulary_)

In [ ]:
import pandas as pd

feature_names = tfidf.get_feature_names_out()

coefficients = model.coef_[0]

top_positive = pd.DataFrame({
    "word": feature_names,
    "weight": coefficients
}).sort_values(
    by="weight",
    ascending=False
)

top_positive.head(20)

In [ ]:
top_negative = pd.DataFrame({
    "word": feature_names,
    "weight": coefficients
}).sort_values(
    by="weight",
    ascending=True
)

top_negative.head(20)

In [ ]:
top_positive.head(20).plot(
    x="word",
    y="weight",
    kind="bar",
    figsize=(12,5)
)

plt.title("Top 20 Positive Words")
plt.show()

In [ ]:
top_negative.head(20).plot(
    x="word",
    y="weight",
    kind="bar",
    figsize=(12,5)
)

plt.title("Top 20 Negative Words")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.show()

In [ ]:
tweet = "This phone is amazing"

tweet = clean_text(tweet)

vector = tfidf.transform([tweet])

print(model.predict(vector))
print(model.predict_proba(vector))

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

# Features and Target
X = df["clean_text"]
y = df["sentiment"]

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        n_jobs=-1
    ),

    "Linear SVC": LinearSVC(),

    # "Naive Bayes": MultinomialNB(),

    # "SGD Classifier": SGDClassifier(
    #     random_state=42
    # )


}

results = []

for name, model in models.items():

    print(f"\n{'='*60}")
    print(f"Training : {name}")
    print(f"{'='*60}")

    pipeline = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50000,
                ngram_range=(1, 3),
                min_df=5,
                max_df=0.95,
                sublinear_tf=True,
                stop_words=None
            )
        ),
        ("model", model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc * 100, 2)
    })

    print(f"\nAccuracy: {acc:.4f}\n")
    print(classification_report(y_test, y_pred))

# Comparison Table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n")
print("="*60)
print("FINAL COMPARISON")
print("="*60)

print(results_df)

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,3)
        )),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    print(
        f"{name}: "
        f"{scores.mean():.4f} "
        f"(+/- {scores.std():.4f})"
    )

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

c_values = [0.1, 0.5, 1, 2, 5, 10]

results = []

for c in c_values:

    pipe = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=100000,
                ngram_range=(1,2),
                min_df=5,
                max_df=0.95,
                sublinear_tf=True
            )
        ),
        (
            "model",
            LogisticRegression(
                C=c,
                max_iter=1000,
                n_jobs=-1
            )
        )
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    results.append((c, acc))

    print(f"C={c}  Accuracy={acc:.4f}")

print("\nFinal Results")
for c, acc in results:
    print(f"C={c} -> {acc:.4f}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

configs = [
    ((1,1), 50000),
    ((1,2), 50000),
    ((1,2), 100000),
    ((1,3), 100000)
]

for ngram, features in configs:

    pipe = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=features,
                ngram_range=ngram,
                min_df=5,
                max_df=0.95,
                sublinear_tf=True
            )
        ),
        (
            "model",
            LogisticRegression(
                C=2,
                max_iter=1000,
                n_jobs=-1
            )
        )
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print(
        f"Ngram={ngram}, "
        f"Features={features}, "
        f"Accuracy={acc:.4f}"
    )

In [ ]:
from sklearn.linear_model import SGDClassifier

pipe = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=100000,
            ngram_range=(1,2),
            min_df=5,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "model",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            random_state=42
        )
    )
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print(
    "SGD Accuracy:",
    accuracy_score(y_test, y_pred)
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report

sgd_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=100000,
            ngram_range=(1,2),
            min_df=5,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "model",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            max_iter=1000,
            random_state=42,
            n_jobs=-1
        )
    )
])

print("Training SGDClassifier...")

sgd_pipeline.fit(X_train, y_train)

y_pred = sgd_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {acc:.4f}\n")

print(classification_report(y_test, y_pred))

In [ ]:
pip install gensim

In [ ]:
import numpy as np
import pandas as pd

from gensim.models import FastText

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:


X_text = df["clean_text"].astype(str)

y = df["sentiment"]

sentences = X_text.apply(lambda x: x.split())

# =========================
# Train FastText
# =========================

print("Training FastText...")

ft_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,          # Skip-Gram
    epochs=10
)



print("FastText Training Complete")

# =========================
# Sentence Embedding Function
# =========================

def get_sentence_embedding(tokens):

    vectors = []

    for word in tokens:

        if word in ft_model.wv:
            vectors.append(ft_model.wv[word])

    if len(vectors) == 0:
        return np.zeros(ft_model.vector_size)

    return np.mean(vectors, axis=0)


# =========================
# Create Feature Matrix
# =========================

print("Creating Embeddings...")

X = np.array(
    [get_sentence_embedding(tokens)
     for tokens in sentences]
)

print("Embedding Shape:", X.shape)



# =========================
# Train Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# Logistic Regression
# =========================

print("Training Logistic Regression...")

model = LogisticRegression(
    C=2,
    max_iter=1000,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Training Complete")


# =========================
# Prediction
# =========================

y_pred = model.predict(X_test)


# =========================
# Evaluation
# =========================

acc = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(acc, 4))

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)


In [25]:
import pandas as pd



df = df[["sentiment", "text"]]

df["sentiment"] = df["sentiment"].replace({
    4:1
})

df = df.sample(
    200000,
    random_state=42
)

print(df.head())
print(df["sentiment"].value_counts())

        sentiment                                               text
541200          0             @chrishasboobs AHHH I HOPE YOUR OK!!! 
750             0  @misstoriblack cool , i have no tweet apps  fo...
766711          0  @TiannaChaos i know  just family drama. its la...
285055          0  School email won't open  and I have geography ...
705995          0                             upper airways problem 
sentiment
1    100143
0     99857
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)



In [27]:
from transformers import AutoTokenizer

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [28]:
import torch

class TwitterDataset(torch.utils.data.Dataset):

    def __init__(self,texts,labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )

        return {
            "input_ids":
            encoding["input_ids"].squeeze(),

            "attention_mask":
            encoding["attention_mask"].squeeze(),

            "labels":
            torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

In [29]:
train_dataset = TwitterDataset(
    train_texts,
    train_labels
)

test_dataset = TwitterDataset(
    test_texts,
    test_labels
)

In [30]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [31]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy":accuracy
    }

In [32]:
from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=64,

    per_device_eval_batch_size=64,

    num_train_epochs=2,

    weight_decay=0.01,

    load_best_model_at_end=True,

    fp16=True
)

In [33]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.285728,0.272250,0.886275
2,0.226795,0.281783,0.889875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5000, training_loss=0.2709182113647461, metrics={'train_runtime': 1099.6765, 'train_samples_per_second': 290.995, 'train_steps_per_second': 4.547, 'total_flos': 1.05244422144e+16, 'train_loss': 0.2709182113647461, 'epoch': 2.0})

In [35]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.226795,0.272250,2,0.886275


{'eval_loss': 0.27224981784820557, 'eval_accuracy': 0.886275}


In [37]:
text = "I absolutely love this movie"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move input tensors to the GPU
inputs = {name: tensor.to('cuda') for name, tensor in inputs.items()}

with torch.no_grad():

    outputs = model(**inputs)

prediction = torch.argmax(
    outputs.logits,
    dim=1
)

if prediction.item()==1:
    print("Positive")
else:
    print("Negative")

Positive


In [38]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4
